# Байесовская оптимизация: поиск оптимума за минимум проб

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Байесовская оптимизация».

## Что делает метод

Байесовская оптимизация ищет оптимум дорогой в вычислении функции за минимальное число проб. Метод чередует два шага. Суррогатная модель (чаще всего гауссовский процесс) приближает целевую функцию и оценивает неопределённость по всему пространству параметров. Функция приобретения превращает прогноз и неопределённость в правило выбора следующей точки для проверки.

Метод применим, когда каждый эксперимент дорог: подбор условий синтеза, настройка установки, калибровка модели. В этом блокноте мы реализуем цикл оптимизации на гауссовском процессе из `scikit-learn`. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы ищете оптимальные условия синтеза нового соединения: температура, давление, концентрация реагента. Каждый опыт занимает день и стоит дорого. Как искать оптимум, не тратя ресурсы вслепую?

Байесовская оптимизация действует как опытный экспериментатор:

1. **После нескольких первых опытов** метод строит **суррогатную модель** — вероятностную карту того, что происходит в пространстве параметров. Суррогатная модель (чаще всего **гауссовский процесс**) — это быстрая замена дорогому эксперименту: она предсказывает результат в любой точке и честно сообщает, где уверена, а где нет.

2. **Функция приобретения** (acquisition function) — правило выбора следующей точки для проверки. Она задаёт баланс между двумя стратегиями:
   - *Эксплуатация* (exploitation): идти туда, где суррогат предсказывает хороший результат.
   - *Разведка* (exploration): идти туда, где суррогат ещё не уверен — вдруг там скрыт оптимум?

3. Цикл «обновить суррогат — выбрать следующую точку — провести опыт» повторяется. Каждый шаг осмысленный, а не случайный.

Ключевое отличие от обычного перебора: метод не тратит пробы на неинтересные области. Он находит оптимум за десятки измерений там, где слепой перебор потребовал бы тысяч.

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q scikit-learn==1.5.1 numpy==1.26.4 pandas==2.2.2 matplotlib==3.9.2 scipy==1.13.1

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

В роли «дорогой» целевой функции возьмём одномерную функцию с несколькими экстремумами и небольшим шумом измерения. Задача — найти её максимум, потратив как можно меньше вычислений. Истинный максимум известен, что позволяет оценить эффективность поиска.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
bounds = (-2.0, 10.0)

def objective(x, noise=0.05):
    """Дорогая целевая функция; цель - найти её максимум."""
    x = np.asarray(x, dtype=float)
    clean = (np.sin(x) + np.sin(0.7 * x) + 0.05 * x)
    return clean + rng.normal(scale=noise, size=np.shape(x))

x_dense = np.linspace(*bounds, 500)
true_clean = np.sin(x_dense) + np.sin(0.7 * x_dense) + 0.05 * x_dense
x_star = x_dense[int(np.argmax(true_clean))]
print(f'Область поиска: {bounds}')
print(f'Истинный максимум вблизи x = {x_star:.2f}, '
      f'значение {true_clean.max():.3f}')

## 4. Применение метода

### Подготовка суррогатной модели и функции приобретения

**Суррогатная модель** — гауссовский процесс из `scikit-learn`. Это вероятностная модель: она возвращает и среднее предсказание, и стандартное отклонение в каждой точке. Стандартное отклонение — мера неопределённости модели в данной точке.

**Ядро** (kernel) — это функция, задающая, насколько похожими должны быть значения функции в двух близких точках. Ядро Матерна (`Matern`) — хороший выбор для реальных физических зависимостей: оно не требует бесконечной гладкости (как RBF), что делает его более реалистичным. `WhiteKernel` добавляет компонент шума измерений.

**Ожидаемое улучшение** (expected improvement, EI) — самая распространённая функция приобретения. Для каждой точки она считает: «если я проведу опыт здесь, каков ожидаемый выигрыш по сравнению с лучшим найденным результатом?». EI велика там, где либо прогноз высок, либо высока неопределённость — и тем самым автоматически уравновешивает разведку и уточнение.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (Matern, WhiteKernel,
                                              ConstantKernel)
from scipy.stats import norm

def make_gp():
    kernel = (ConstantKernel(1.0) * Matern(length_scale=1.0,
              nu=2.5) + WhiteKernel(noise_level=0.01))
    return GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                    n_restarts_optimizer=5,
                                    random_state=42)

def expected_improvement(x, gp, best_y, xi=0.01):
    """Ожидаемое улучшение для задачи максимизации."""
    mu, sigma = gp.predict(x.reshape(-1, 1), return_std=True)
    sigma = np.maximum(sigma, 1e-9)
    imp = mu - best_y - xi
    z = imp / sigma
    return imp * norm.cdf(z) + sigma * norm.pdf(z)

print('Суррогатная модель и функция приобретения готовы.')

### Цикл оптимизации

Начнём с трёх случайных проб (холодный старт) и выполним 12 шагов байесовской оптимизации. На каждом шаге:
1. Гауссовский процесс обучается на накопленных пробах.
2. Вычисляется функция приобретения (EI) по всем точкам сетки.
3. Выбирается точка с наибольшим EI — следующий «опыт».
4. Результат добавляется в набор проб.

Мы сохраняем историю лучшего результата, чтобы потом построить кривую сходимости.

In [ ]:
x_sampled = list(rng.uniform(*bounds, size=3))
y_sampled = list(objective(x_sampled))

n_steps = 12
history_best = [max(y_sampled)]
for step in range(n_steps):
    gp = make_gp()
    gp.fit(np.array(x_sampled).reshape(-1, 1), np.array(y_sampled))
    ei = expected_improvement(x_dense, gp, max(y_sampled))
    x_next = x_dense[int(np.argmax(ei))]
    y_next = float(objective(x_next))
    x_sampled.append(x_next)
    y_sampled.append(y_next)
    history_best.append(max(y_sampled))

best_idx = int(np.argmax(y_sampled))
print(f'Проведено проб: {len(x_sampled)}')
print(f'Найденный максимум: x = {x_sampled[best_idx]:.2f}, '
      f'значение {y_sampled[best_idx]:.3f}')
print(f'Истинный максимум вблизи x = {x_star:.2f}')

### Визуализация: суррогатная модель и ход оптимизации

Для наглядности покажем финальное состояние суррогатной модели и пошаговую демонстрацию первых шагов оптимизации — как прогноз и полоса неопределённости меняются с каждой новой пробой.

In [ ]:
import matplotlib.pyplot as plt

# --- Финальное состояние суррогата и кривая сходимости ---
gp_final = make_gp()
gp_final.fit(np.array(x_sampled).reshape(-1, 1), np.array(y_sampled))
mu, sigma = gp_final.predict(x_dense.reshape(-1, 1), return_std=True)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))

# Левый: суррогатная модель
axes[0].plot(x_dense, true_clean, color=VIZ['series'][2],
             linestyle='--', label='целевая функция (скрыта от метода)')
axes[0].plot(x_dense, mu, color=VIZ['series'][0],
             label='прогноз суррогата')
axes[0].fill_between(x_dense, mu - 2 * sigma, mu + 2 * sigma,
                     color=VIZ['series'][0], alpha=0.2,
                     label='неопределённость суррогата')
axes[0].scatter(x_sampled[:3], y_sampled[:3], color=VIZ['series'][3],
                marker='s', s=80, zorder=6, label='стартовые пробы (3 шт.)')
axes[0].scatter(x_sampled[3:], y_sampled[3:], color=VIZ['series'][1],
                zorder=5, label='пробы байесовской оптимизации')
axes[0].axvline(x_star, color=VIZ['series'][2], linestyle=':',
                label='истинный максимум')
axes[0].set_xlabel('Параметр x')
axes[0].set_ylabel('Значение функции')
axes[0].set_title(f'Суррогатная модель после {len(x_sampled)} проб')
axes[0].legend(fontsize=9)

# Правый: сходимость
axes[1].plot(range(len(history_best)), history_best, marker='o',
             color=VIZ['series'][0])
axes[1].axhline(true_clean.max(), color=VIZ['series'][2],
                linestyle='--', label='истинный максимум')
axes[1].set_xlabel('Число проб (от начала)')
axes[1].set_ylabel('Лучшее найденное значение')
axes[1].set_title('Сходимость байесовской оптимизации')
axes[1].legend()

fig.tight_layout()
plt.show()

# --- Пошаговая иллюстрация: как меняется суррогат с каждой пробой ---
# Берём первые 3, 5, 8, 15 проб и рисуем состояние суррогата
snapshots = [3, 5, 8, 15]
fig2, axes2 = plt.subplots(1, len(snapshots), figsize=(15.0, 4.2),
                            sharey=True)
for ax, n_pts in zip(axes2, snapshots):
    xs = np.array(x_sampled[:n_pts]).reshape(-1, 1)
    ys = np.array(y_sampled[:n_pts])
    gp_snap = make_gp()
    gp_snap.fit(xs, ys)
    m_s, s_s = gp_snap.predict(x_dense.reshape(-1, 1), return_std=True)
    ax.plot(x_dense, true_clean, color=VIZ['series'][2],
            linestyle='--', linewidth=1.5)
    ax.plot(x_dense, m_s, color=VIZ['series'][0])
    ax.fill_between(x_dense, m_s - 2 * s_s, m_s + 2 * s_s,
                    color=VIZ['series'][0], alpha=0.25)
    ax.scatter(x_sampled[:n_pts], y_sampled[:n_pts],
               color=VIZ['series'][3], zorder=5, s=45)
    ax.axvline(x_star, color=VIZ['series'][2], linestyle=':', linewidth=1)
    ax.set_title(f'{n_pts} проб')
    ax.set_xlabel('Параметр x')
axes2[0].set_ylabel('Значение функции')
fig2.suptitle('Сужение неопределённости суррогата с ростом числа проб',
              fontsize=13, y=1.02)
fig2.tight_layout()
plt.show()

**Как читать эти графики.**

*Верхний левый — финальная суррогатная модель.* Синяя линия — прогноз суррогата; голубая полоса — неопределённость (±2 стандартных отклонения). Пунктирная линия — истинная функция, скрытая от метода. Квадратные точки — 3 стартовых случайных пробы; круглые — пробы, выбранные байесовской оптимизацией. Пунктирная вертикаль — истинный максимум. Видно: метод сконцентрировал пробы вблизи максимума, а суррогат хорошо приближает истинную функцию.

*Верхний правый — кривая сходимости.* Горизонтальная ось — номер пробы (всего 15 = 3 старт + 12 шагов). Вертикальная ось — лучшее найденное значение на каждом шаге. Кривая должна стабилизироваться вблизи истинного максимума за несколько десятков шагов.

*Нижняя строка из 4 панелей — как сужается полоса неопределённости по мере накопления проб.* При 3 пробах полоса широкая — мы почти ничего не знаем. При 15 пробах суррогат уверенно восстанавливает форму функции в области максимума.

## 5. Интерпретация результата

- **Прогноз суррогатной модели**: гауссовский процесс приближает целевую функцию по немногим пробам; **полоса неопределённости** показывает, где модель не уверена. Широкая полоса — сигнал «здесь нужна проба».
- **Расположение проб**: метод концентрирует пробы возле перспективных областей, но изредка проверяет и неисследованные участки — это баланс **разведки** (exploration) и **уточнения** (exploitation). Если бы метод только «уточнял», он мог бы застрять в локальном максимуме.
- **Функция приобретения EI**: параметр `xi` управляет балансом — чем он больше, тем агрессивнее разведка. Значение по умолчанию `xi=0.01` — хороший старт.
- **Кривая сходимости**: лучшее найденное значение быстро приближается к истинному максимуму; выход кривой на плато означает, что дальнейшие пробы малополезны.
- Эффективность зависит от выбора ядра суррогатной модели; для зашумлённых и многоэкстремальных функций разумно повторять оптимизацию с разных случайных стартов.

## 6. Попробуйте на своей задаче

Замените демонстрационную функцию своей целевой функцией. Это может быть результат эксперимента, расчёта или симуляции.

1. Снимите комментарии в ячейке ниже и опишите свою функцию и границы поиска.
2. Если каждая проба — реальный эксперимент, выполняйте цикл пошагово, подставляя измеренные значения вручную.
3. Выполните ячейки раздела 4.

In [ ]:
# --- Шаблон для своей задачи ---
# bounds = (нижняя_граница, верхняя_граница)
#
# def objective(x):
#     # верните измеренное или вычисленное значение в точке x;
#     # метод ищет максимум - при минимизации верните значение со
#     # знаком минус
#     return ...
#
# x_dense = np.linspace(*bounds, 500)
# Далее повторите ячейки раздела 4.

## Краткие выводы

- Байесовская оптимизация находит оптимум **дорогой функции** за минимальное число проб, чередуя два шага: обновление суррогатной модели и выбор следующей точки по функции приобретения.
- **Суррогатная модель** (гауссовский процесс) — это вероятностная карта: прогноз + неопределённость по всему пространству параметров.
- **Функция приобретения** (EI) автоматически уравновешивает разведку (исследование незнакомых областей) и уточнение (опыты вблизи известного оптимума).
- Метод особенно ценен в науке, когда каждый эксперимент дорог: синтез материала, постановка опыта, прогон тяжёлой симуляции.
- За 10–20 шагов метод обычно находит оптимум там, где слепой перебор потребовал бы сотен или тысяч проб.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На графике суррогатной модели полоса неопределённости в некоторых областях остаётся широкой даже после 15 проб. Что это означает с точки зрения дальнейшей стратегии: стоит ли проводить пробы именно в этих областях, и при каком условии это имеет смысл?
2. Кривая сходимости вышла на плато после 10-й пробы, не достигнув истинного максимума. Перечислите два возможных объяснения: одно связанное с функцией приобретения, другое — с выбором ядра суррогатной модели.
3. Вы хотите применить байесовскую оптимизацию к задаче с 10 параметрами вместо одного. Почему эффективность метода снижается с ростом размерности пространства поиска, и какой практический порог числа параметров принято считать приемлемым?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Широкая полоса в области без проб — сигнал неисследованного пространства. Проводить пробу здесь имеет смысл, если функция приобретения (EI) указывает на высокий ожидаемый выигрыш — то есть если неопределённость сочетается с достаточно высоким предсказанием суррогата. Если суррогат предсказывает там низкие значения с большой неопределённостью, проба нецелесообразна при ограниченном бюджете.
2. Со стороны функции приобретения: малое `xi` делает поиск слишком «жадным» (exploitation), и метод застревает в локальном максимуме, не исследуя другие пики. Со стороны ядра: если длина корреляции ядра (length_scale) плохо подобрана под реальный масштаб изменчивости функции, суррогат неверно обобщает между пробами — либо «перегладит», либо будет слишком локальным.
3. Гауссовский процесс страдает от «проклятия размерности»: объём пространства экспоненциально растёт с числом параметров, и то же число проб покрывает его всё хуже. Кроме того, подбор гиперпараметров ядра становится вычислительно дороже. На практике байесовская оптимизация эффективна до ~20 параметров; при большем числе применяют случайный поиск, эволюционные алгоритмы или методы с разложением пространства (CMA-ES, TuRBO).
</details>


## 7. Поэкспериментируйте сами

Попробуйте изменить следующие параметры и посмотрите, что изменится:

- **`n_steps`** (количество шагов оптимизации, раздел 4): увеличьте до 20 или уменьшите до 5. При каком числе шагов оптимизация надёжно находит максимум?
- **`xi`** в функции `expected_improvement`: попробуйте `xi=0.001` (жадное уточнение) и `xi=0.5` (активная разведка). Как меняется расположение проб?
- **Начальный размер**: измените `size=3` на `size=1` или `size=8` в строке генерации стартовых проб. Как это влияет на скорость сходимости?
- **Целевая функция**: замените функцию `objective` на `lambda x: -np.sin(x) * np.cos(x) + 0.1 * x` (другой ландшафт с несколькими экстремумами). Справляется ли метод за то же число проб?